<a href="https://colab.research.google.com/github/fardeenshroff/lenet5-gpu-autotvm-optimization/blob/main/notebooks/gpu_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Five-Epoch NVIDIA GPU Benchmark: LeNet-5 on Google Colab

**Owner:** [Fardeen Shroff](https://github.com/fardeenshroff)  |  **Repository:** [lenet5-gpu-autotvm-optimization](https://github.com/fardeenshroff/lenet5-gpu-autotvm-optimization)

**Before running:** Choose **Runtime → Change runtime type → T4 GPU** (or any other NVIDIA GPU that Colab offers), then **Runtime → Run all**.

The notebook verifies **real NVIDIA GPU execution** by (1) detecting the GPU, (2) performing a CUDA matrix multiplication, (3) validating its result against CPU, (4) training LeNet-5 for **five epochs** on MNIST using CUDA and collecting per-epoch test accuracy, and (5) measuring matched-batch CUDA vs CPU inference on the *same trained model*. It also attempts a standalone compiled CUDA/C++ convolution test and exports the raw machine-readable results.

**Metric scope:** Newly measured PyTorch inference speedups are not the same as the original report's **2.39× training** comparison. No historical result is considered reproduced without a matched original protocol.


In [9]:
import sys, os, subprocess, pathlib, shutil, json, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No NVIDIA GPU connected. Select Runtime > Change runtime type > T4 GPU and reconnect.")
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("nvidia-smi not found; CUDA runtime will still be tested below.")


Python: 3.13.16
PyTorch: 2.14.1+cu130
PyTorch CUDA runtime: 13.0
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)


## 1. Install and test the public GitHub project

The notebook pulls the current public code instead of storing a second copy of the implementation.


In [10]:
repo_dir = pathlib.Path("/content/lenet5-gpu-autotvm-optimization")
url = "https://github.com/fardeenshroff/lenet5-gpu-autotvm-optimization.git"
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[test]"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## 2. Verify actual CUDA hardware execution

A GPU check is considered **passed** only if CUDA is accessible, a matrix multiplication executes on the GPU, and its numeric output agrees with the CPU reference. A JSON report preserves the hardware identification and measured CUDA event timings.


In [11]:
subprocess.run([sys.executable, "scripts/verify_cuda_runtime.py",
                "--output", "results/colab_gpu_runtime_verification_5epochs.json"], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/verify_cuda_runtime.py', '--output', 'results/colab_gpu_runtime_verification_5epochs.json'], returncode=0)

## 3. Train LeNet-5 for five epochs on MNIST with NVIDIA CUDA

This produces a **new** model checkpoint and fresh test-set accuracy. It is not a reproduction of the original training-time benchmark, because the training configuration and hardware may differ.


In [12]:
subprocess.run([sys.executable, "scripts/train.py", "--device", "cuda",
                "--epochs", "5", "--batch-size", "256",
                "--output", "checkpoints/colab_cuda_5epochs.pt",
                "--metrics", "results/colab_cuda_training_5epochs.json"], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/train.py', '--device', 'cuda', '--epochs', '5', '--batch-size', '256', '--output', 'checkpoints/colab_cuda_5epochs.pt', '--metrics', 'results/colab_cuda_training_5epochs.json'], returncode=0)

## 3b. Matched PyTorch CPU training baseline

Run the same 5-epoch MNIST configuration on the **same Colab host CPU** to calculate a **new PyTorch CPU-versus-CUDA training comparison**. This is not a TVM/AutoTVM optimization measurement.


In [13]:
subprocess.run([sys.executable, "scripts/train.py", "--device", "cpu",
                "--epochs", "5", "--batch-size", "256",
                "--seed", "42", "--output", "checkpoints/colab_cpu_5epochs.pt",
                "--metrics", "results/colab_cpu_training_5epochs.json"], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/train.py', '--device', 'cpu', '--epochs', '5', '--batch-size', '256', '--seed', '42', '--output', 'checkpoints/colab_cpu_5epochs.pt', '--metrics', 'results/colab_cpu_training_5epochs.json'], returncode=0)

## 4. Compare CPU and NVIDIA GPU inference latency

Both backends evaluate the same checkpoint and identical input shapes in the **same Colab session**. Each test includes warmup and measures synchronized forward-pass latency; small batches may be slower on a GPU because of dispatch overhead.


In [14]:
from pathlib import Path
for batch_size in (1, 128):
    for device in ("cpu", "cuda"):
        output = f"results/colab_{device}_inference_5epochs_batch{batch_size}.json"
        subprocess.run([
            sys.executable, "scripts/benchmark_pytorch.py",
            "--device", device, "--checkpoint", "checkpoints/colab_cuda_5epochs.pt",
            "--batch-size", str(batch_size), "--warmup", "30",
            "--iterations", "100", "--output", output
        ], check=True)


## 5. Compile and verify a standalone CUDA/C++ convolution kernel

This is a **separate educational kernel**, not the kernel generated in the historical study. If the Colab image does not have `nvcc`, the compiled-kernel status is recorded as *unavailable* rather than claimed as successful.


In [15]:
cuda_kernel = {"status": "not_run"}
if shutil.which("nvcc"):
    compile_run = subprocess.run(
        ["nvcc", "-O3", "-std=c++14", "cuda/conv2d_naive.cu",
         "-o", "/tmp/conv2d_naive"], capture_output=True, text=True
    )
    if compile_run.returncode == 0:
        executed = subprocess.run(["/tmp/conv2d_naive"], capture_output=True, text=True)
        cuda_kernel = {"status": "passed" if executed.returncode == 0 else "failed",
                       "exit_code": executed.returncode, "stdout": executed.stdout,
                       "stderr": executed.stderr}
        print(executed.stdout, executed.stderr)
    else:
        cuda_kernel = {"status": "compile_failed",
                       "exit_code": compile_run.returncode,
                       "stdout": compile_run.stdout, "stderr": compile_run.stderr}
        print(compile_run.stderr)
else:
    cuda_kernel = {"status": "nvcc_not_available"}
    print("nvcc is unavailable on this runtime; no compiled-kernel claim is made.")
Path("results/colab_cuda_cpp_kernel_5epochs.json").write_text(
    json.dumps(cuda_kernel, indent=2) + "\n")


Average CUDA kernel time (ms): 0.00616448
Max absolute error vs CPU: 9.53674e-07
 


158

## 6. Summarize the measured speedups and download the evidence

The summary embeds the **measured** machine/runtime details, checkpoint context, latency ratios, and raw result filenames. Your browser downloads a ZIP so the results can be independently inspected and published to GitHub.


In [16]:
import zipfile, datetime
root = pathlib.Path("results")
probe = json.loads((root / "colab_gpu_runtime_verification_5epochs.json").read_text())
training = json.loads((root / "colab_cuda_training_5epochs.json").read_text())
cpu_training = json.loads((root / "colab_cpu_training_5epochs.json").read_text())
comparisons = []
for batch_size in (1, 128):
    cpu = json.loads((root / f"colab_cpu_inference_5epochs_batch{batch_size}.json").read_text())
    gpu = json.loads((root / f"colab_cuda_inference_5epochs_batch{batch_size}.json").read_text())
    ratio = cpu["median_ms"] / gpu["median_ms"]
    comparisons.append({
        "batch_size": batch_size, "cpu_median_ms": cpu["median_ms"],
        "gpu_median_ms": gpu["median_ms"],
        "cpu_to_gpu_inference_speedup": ratio, "iterations": cpu["iterations"]
    })
summary = {
    "status": "completed_with_actual_cuda_execution",
    "run_time_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source_repo": url,
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "epochs": 5,
    "gpu": probe["device_name"], "cuda_runtime_verified": probe["test_passed"],
    "cuda_training_accuracy_percent": 100 * training["final_test_accuracy"],
    "cuda_training_duration_seconds": training["total_wall_seconds"],
    "cpu_training_accuracy_percent": 100 * cpu_training["final_test_accuracy"],
    "cpu_training_duration_seconds": cpu_training["total_wall_seconds"],
    "pytorch_cpu_to_cuda_training_speedup": (
        cpu_training["total_wall_seconds"] / training["total_wall_seconds"]
    ),
    "training_comparison_method": "five epochs, PyTorch both devices, same host and batch size; not AutoTVM",
    "inference_comparisons": comparisons,
    "cuda_cpp_kernel": cuda_kernel,
    "historical_2_39x_training_speedup_reproduced": False,
}
summary_path = root / "colab_verified_gpu_5epochs_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))
archive = pathlib.Path("/content/fardeen_lenet5_colab_5epochs_gpu_results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in root.glob("colab_*_5epochs*.json"):
        z.write(file, arcname=file.name)
print("Saved GPU benchmark evidence:", archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Colab download helper unavailable; retrieve the archive from", archive)


{
  "status": "completed_with_actual_cuda_execution",
  "run_time_utc": "2026-11-22T17:24:27.702490+00:00",
  "source_repo": "https://github.com/fardeenshroff/lenet5-gpu-autotvm-optimization.git",
  "git_commit": "d0be4d471699153fe69c395f336ab52375a9402c",
  "epochs": 5,
  "gpu": "Tesla T4",
  "cuda_runtime_verified": true,
  "cuda_training_accuracy_percent": 98.52,
  "cuda_training_duration_seconds": 82.90360534700005,
  "cpu_training_accuracy_percent": 98.58,
  "cpu_training_duration_seconds": 122.27367926799991,
  "pytorch_cpu_to_cuda_training_speedup": 1.4748897681375024,
  "training_comparison_method": "five epochs, PyTorch both devices, same host and batch size; not AutoTVM",
  "inference_comparisons": [
    {
      "batch_size": 1,
      "cpu_median_ms": 0.44938150006146316,
      "gpu_median_ms": 0.34864800011291663,
      "cpu_to_gpu_inference_speedup": 1.2889260799313977,
      "iterations": 100
    },
    {
      "batch_size": 128,
      "cpu_median_ms": 7.428717000038887,
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>